# Walk pilot on larger models: Gemma 3 27B, Qwen3-32B, Llama 3.3 70B (preregistered: `docs/preregistration_walk_pilot.md` + `docs/preregistration_walk_ladder.md`)

The same 2,400 walks as the 12B pilot, scored by three larger models one after another. Answers only. Each model is
deleted from the Colab disk once it is done, to make room for the next.

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
   For Llama, the same Hugging Face account must have access to `meta-llama/Llama-3.3-70B-Instruct`; the notebook
   checks it and skips Llama with a message if not.
3. **Runtime → Change runtime type → G4 (RTX PRO 6000, 96 GB)** if offered, otherwise **H100** or **A100 80 GB**.
   A 40 GB A100 is not enough.

Then **Runtime → Run all**. Expected time: about 2–3 hours, mostly downloads (54 GB, 64 GB, then 140 GB for Llama).
If Colab disconnects, press **Run all** again; finished models and finished scoring passes are skipped.


In [ ]:
# 1. GPU and disk check
import subprocess, torch, os, shutil
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> G4 / H100 / A100 80 GB"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 70, f"Only {gb:.0f} GB of GPU memory: pick G4, H100 or A100 80 GB"
free = shutil.disk_usage("/content").free / 1e9
print(f"GPU {gb:.0f} GB, free disk {free:.0f} GB")
assert free > 160, f"Only {free:.0f} GB free disk: Llama 70B needs about 145 GB"


In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/docs/preregistration_walk_ladder.md"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
print("already saved:", sorted(os.listdir(f"{RES}/walks")) if os.path.exists(f"{RES}/walks") else "nothing yet")

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import); bitsandbytes for Llama 70B
!pip -q install "transformers==4.57.3" nnsight accelerate bitsandbytes
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"


In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional); Llama access check
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami, hf_hub_download
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
try:
    hf_hub_download("meta-llama/Llama-3.3-70B-Instruct", "config.json", token=os.environ["HF_TOKEN"])
    LLAMA = True
    print("Llama 3.3 70B: access OK")
except Exception as e:
    LLAMA = False
    print("Llama 3.3 70B: NO ACCESS with this token, it will be skipped. Request it at "
          "https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct and rerun later.", type(e).__name__)
import sys
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Walk ladder] Colab run started (or resumed)" + ("" if LLAMA else "; no Llama access"))


In [ ]:
# 5. The three models, one after another (each resumes if interrupted; each is deleted from disk when done)
import glob
RUNS = [("gemma3-27b", "google/gemma-3-27b-it", 16), ("qwen3-32b", "Qwen/Qwen3-32B", 16)]
if LLAMA:
    RUNS.append(("llama3.3-70b", "meta-llama/Llama-3.3-70B-Instruct", 8))
for key, repo, bs in RUNS:
    print(f"===== {key} =====", flush=True)
    !cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/walk_pilot.py --model {key} --batch-size {bs} 2>&1 | grep -v -i warning
    done = glob.glob(f"{RES}/walks/{key}_walk_pilot_*.done")
    print(f"{key}: {'done: ' + open(done[0]).read() if done else 'NOT finished (see the output above)'}")
    shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{repo.replace('/', '--')}"), ignore_errors=True)


In [ ]:
# 6. One zip of the results + the results log, on Drive and downloaded here
with zipfile.ZipFile("/content/walk_ladder_results.zip", "w") as z:
    for key, _, _ in RUNS:
        for f in glob.glob(f"{RES}/walks/{key}_walk_pilot*"):
            z.write(f, "results/walks/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
shutil.copy("/content/walk_ladder_results.zip", f"{D}/walk_ladder_results.zip")
notify("[Walk ladder] all done: results in MyDrive/geomap/walk_ladder_results.zip")
from google.colab import files
files.download("/content/walk_ladder_results.zip")
